# Advanced Artificial Intelligence Task 2
### Produce classification via pre-trained models across different architectures

- **CNN**:          EfficientNet_V2_S_Weights.IMAGENET1K_V1
- **TRANSFORMER**:   Swin_S_Weights.IMAGENET1K_V1 
- **HYBRID**:       MaxVit_T
Details for the pre-trained weights can be found [here](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.efficientnet_v2_s.html#torchvision.models.efficientnet_v2_s).

View logs with ```tensorboard --logdir task_2/runs```.

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import os
import torch.optim as optim
import torchmetrics
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
from pathlib import Path
import datetime
from torchvision.models import get_model, get_weight
import sys
from sklearn.model_selection import train_test_split

sys.path.append("..")
sys.path.append(".")
from experiment_configs import task_2_config as experiments
from utils.dataset import ProduceDataset

PRODUCE_DATASET_PATH = Path(".") / "data" / "Fruit_And_Vegetable_Diseases_Dataset_no_identical"
WORKERS = int(os.cpu_count() * 0.20)
VAL_WORKERS = WORKERS

# Select and print available experiments (defined in experiment_configs/task_2_config.py)
print(f"Selectable experiments: {[name for name in dir(experiments) if isinstance(getattr(experiments, name), experiments.Experiment)]}")
EXPERIMENT = experiments.efficientnet_finetune_adam_cw
print("\nSELECTED: ", EXPERIMENT.training)

# Unpack experiment hyperparameters
TRANSFER_TYPE = EXPERIMENT.training.transfer_type
LEARNING_RATE = EXPERIMENT.training.learning_rate
MOMENTUM      = EXPERIMENT.training.momentum
AUGMENT       = EXPERIMENT.training.augment
WEIGHT_DECAY  = EXPERIMENT.training.weight_decay

# Gradient accumulation: keep effective batch size = 32 regardless of GPU memory
BATCH_SIZE         = 8
ACCUMULATION_STEPS = 4  # effective = BATCH_SIZE * ACCUMULATION_STEPS
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * ACCUMULATION_STEPS
assert EFFECTIVE_BATCH_SIZE == 32, "Effective batch size must be 32. Adjust BATCH_SIZE or ACCUMULATION_STEPS."

TEST_SPLIT             = 0.8
MAXIMUM_EPOCHS         = 20  # upper bound; early stopping will typically terminate sooner
NUM_CLASSES            = 2   # Healthy vs Rotten
EARLY_STOPPING_PATIENCE = 5

# Load pre-trained weights and extract the required input transforms for the chosen architecture
pretrained_weights = get_weight(EXPERIMENT.weights)
PRETRAINED_MODEL   = get_model(EXPERIMENT.architecture, weights=pretrained_weights)
auto_transforms    = pretrained_weights.transforms()
print(auto_transforms)

Selectable experiments: ['efficientnet_finetune_adam_cw', 'efficientnet_finetune_adam_cw_noaug', 'efficientnet_finetune_augment_cw', 'efficientnet_finetune_baseline', 'efficientnet_freeze_baseline']

SELECTED:  Training(transfer_type='FINETUNE', learning_rate=0.0001, momentum=0.9, batch_size=32, max_epochs=10, augment=True, optimizer='Adam', weight_decay=0.0001, class_weights=True)
ImageClassification(
    crop_size=[384]
    resize_size=[384]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [ ]:
# Load dataset — labels are assigned from folder names (Category__Split convention)
produce_dataset = ProduceDataset(root_dir=PRODUCE_DATASET_PATH, transform=auto_transforms)
produce_dataset.print_class_balance()
produce_dataset.display_examples(num_samples=5, show_transformed=False)

print(f"\nLoaded {len(produce_dataset)} images total\n")
print(f"{'Folder':<30} {'Count':>6}")
print("-" * 38)
for folder in sorted(PRODUCE_DATASET_PATH.iterdir()):
    if folder.is_dir():
        count = len(list(folder.glob("*.jpg"))) + len(list(folder.glob("*.png")))
        print(f"{folder.name:<30} {count:>6}")

# Transfer Learning vs. Finetuning
Both methods implement a classifer head (the final layer) for the new task (Healthy vs. Rotten). They adapt pre-trained model behaviour in distinct ways:
| Method | Description | When to use? |
|---|---|---|
| Transfer learning | Freeze most or all pre-trained weights and train new classifer head. | Small dataset, similar domain and task. |
| Fine-tuning | Update some or all model weights as well as new classifier head.| Large dataset, less similar domain and task |


In [ ]:
# Freeze backbone weights for transfer learning; leave all weights trainable for fine-tuning
if TRANSFER_TYPE == "FREEZE":
    for parameters in PRETRAINED_MODEL.parameters():
        parameters.requires_grad = False

# Replace the 1000-class ImageNet head with a 2-class (Healthy / Rotten) head
num_features = PRETRAINED_MODEL.classifier[1].in_features
PRETRAINED_MODEL.classifier[1] = nn.Linear(num_features, NUM_CLASSES)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = PRETRAINED_MODEL.to(device)

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Optimizer — FREEZE trains only the new head; FINETUNE updates all parameters
if EXPERIMENT.training.optimizer == "SGD":
    params = PRETRAINED_MODEL.classifier.parameters() if TRANSFER_TYPE == "FREEZE" else PRETRAINED_MODEL.parameters()
    optimizer = optim.SGD(params, lr=LEARNING_RATE, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
else:  # Adam
    params = PRETRAINED_MODEL.classifier.parameters() if TRANSFER_TYPE == "FREEZE" else PRETRAINED_MODEL.parameters()
    optimizer = optim.Adam(params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# LR scheduler: multiply LR by gamma every step_size epochs
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=EXPERIMENT.scheduler.step_size,
    gamma=EXPERIMENT.scheduler.gamma,
)

In [ ]:
from torch.utils.data import WeightedRandomSampler
from collections import Counter

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Stratify by produce category + label so the split preserves per-category class balance
stratify_keys = [
    os.path.basename(os.path.dirname(produce_dataset.image_paths[i])).split("__")[0]
    + "_" + str(produce_dataset.labels[i])
    for i in range(len(produce_dataset))
]

train_idx, val_idx = train_test_split(
    range(len(produce_dataset)),
    test_size=1 - TEST_SPLIT,
    stratify=stratify_keys,
    random_state=42,
)

# Class-weighted loss to counteract Healthy/Rotten imbalance (computed on training set only)
if EXPERIMENT.training.class_weights:
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(produce_dataset.labels),
        y=[produce_dataset.labels[i] for i in train_idx],
    )
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float).to(device))
    print(f"Class weights: {class_weights}")
else:
    criterion = nn.CrossEntropyLoss()

# Separate dataset instances: training uses a fresh copy so GPU augmentation is applied independently
train_produce_dataset = ProduceDataset(root_dir=PRODUCE_DATASET_PATH, transform=auto_transforms)
train_dataset = torch.utils.data.Subset(train_produce_dataset, train_idx)
val_dataset   = torch.utils.data.Subset(produce_dataset, val_idx)

# WeightedRandomSampler: up-sample underrepresented produce categories (Grape, Jujube, Pomegranate …)
# so every category sees roughly equal exposure per epoch
train_produces = [
    os.path.basename(os.path.dirname(train_produce_dataset.image_paths[i])).split("__")[0]
    for i in train_idx
]
produce_counts = Counter(train_produces)
sample_weights = torch.tensor([1.0 / produce_counts[p] for p in train_produces])
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

print("Produce category counts in training set:")
for produce, count in sorted(produce_counts.items()):
    print(f"  {produce:<15} {count} images")

# shuffle=False required when a sampler is provided
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=VAL_WORKERS, pin_memory=VAL_WORKERS > 0)

print(f"\nTraining size: {len(train_dataset)}  |  Validation size: {len(val_dataset)}")

In [ ]:
from torchvision.transforms import v2 as T

# ── Helper: running average ──────────────────────────────────────────────────
class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.sum = 0
        self.count = 0

    def update(self, value, n=1):
        self.sum += value * n
        self.count += n

    @property
    def avg(self):
        return self.sum / self.count if self.count > 0 else 0


# ── Training loop ────────────────────────────────────────────────────────────
def train_one_epoch(model, dataloader, criterion, optimizer, device, epoch, accuracy_metric, accumulation_steps=1, augment=None):
    model.train()
    loss_meter = AverageMeter()
    accuracy_metric.reset()
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Training]", leave=True)

    optimizer.zero_grad()
    for id, (X_batch, y_batch) in enumerate(progress_bar):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        if augment is not None:
            X_batch = augment(X_batch)
            X_batch = X_batch.clamp(-3.0, 3.0)

        # Scale loss by accumulation steps so gradient magnitude is independent of BATCH_SIZE
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch) / accumulation_steps
        loss.backward()

        if (id + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        loss_meter.update(loss.item() * accumulation_steps, X_batch.size(0))
        preds = outputs.argmax(dim=1)
        accuracy_metric.update(preds, y_batch)

        if id % 50 == 0:
            progress_bar.set_postfix(loss=loss_meter.avg, accuracy=accuracy_metric.compute().item())

    # Flush remaining gradients if the last mini-batch didn't land on an accumulation boundary
    if (id + 1) % accumulation_steps != 0:
        optimizer.step()
        optimizer.zero_grad()

    return loss_meter.avg, accuracy_metric.compute().item()


# ── Validation loop ──────────────────────────────────────────────────────────
def validate(model, dataloader, criterion, device, epoch, accuracy_metric):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    accuracy_metric.reset()
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Validation]", leave=True)

    with torch.no_grad():
        for id, (X_batch, y_batch) in enumerate(progress_bar):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            total_samples += X_batch.size(0)
            preds = outputs.argmax(dim=1)
            accuracy_metric.update(preds, y_batch)
            if id % 50 == 0:
                progress_bar.set_postfix(loss=total_loss / total_samples, accuracy=accuracy_metric.compute().item())

    return total_loss / total_samples, accuracy_metric.compute().item()


# ── GPU augmentation pipeline ────────────────────────────────────────────────
# Applied on-the-fly on the GPU batch; never written to disk.
gpu_augment = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandAugment(num_ops=2, magnitude=6),
])


In [ ]:
# ── Augmentation visualisation ───────────────────────────────────────────────
# Run this cell to inspect what the model actually sees after augmentation.
# Uses raw PIL images (no normalisation) so colours are true to the original.
from PIL import Image
import matplotlib.pyplot as plt
from torchvision.transforms import v2 as T
import random

raw_paths = random.sample(train_produce_dataset.image_paths, 4)
to_tensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Resize((384, 384))])

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Top: original  |  Bottom: after augmentation", fontsize=11)

for i, path in enumerate(raw_paths):
    img    = Image.open(path).convert("RGB")
    tensor = to_tensor(img)

    axes[0, i].imshow(tensor.permute(1, 2, 0).numpy())
    axes[0, i].set_title(Path(path).parent.name, fontsize=7)
    axes[0, i].axis("off")

    augmented = gpu_augment(tensor.unsqueeze(0)).squeeze(0)
    axes[1, i].imshow(augmented.clamp(0, 1).permute(1, 2, 0).numpy())
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ── Training run ─────────────────────────────────────────────────────────────
writer = SummaryWriter()

train_accuracy = torchmetrics.Accuracy(task="binary", num_classes=NUM_CLASSES).to(device)
val_accuracy   = torchmetrics.Accuracy(task="binary", num_classes=NUM_CLASSES).to(device)

timestamp      = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
model_save_name = f"{EXPERIMENT.display_name}_{timestamp}"

print(f"Training on: {device}  |  Effective batch size: {EFFECTIVE_BATCH_SIZE}  |  Accumulation steps: {ACCUMULATION_STEPS}")

best_val_loss    = float("inf")
patience_counter = 0
start_epoch      = 0

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()

# To resume from a checkpoint, uncomment and set the path:
# checkpoint = torch.load("models/your_model.pth", weights_only=False)
# PRETRAINED_MODEL.load_state_dict(checkpoint["model_state_dict"])
# optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
# lr_scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
# best_val_loss    = checkpoint["best_val_loss"]
# patience_counter = checkpoint["patience_counter"]
# start_epoch      = checkpoint["epoch"] + 1

for epoch in range(start_epoch, MAXIMUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        PRETRAINED_MODEL, train_loader, criterion, optimizer, device, epoch,
        train_accuracy, ACCUMULATION_STEPS, gpu_augment if AUGMENT else None,
    )

    val_loss, val_acc = validate(PRETRAINED_MODEL, val_loader, criterion, device, epoch, val_accuracy)

    lr_scheduler.step()

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f} | Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

    writer.add_scalar("Loss/Train",        train_loss, epoch)
    writer.add_scalar("Accuracy/Train",    train_acc,  epoch)
    writer.add_scalar("Loss/Validation",   val_loss,   epoch)
    writer.add_scalar("Accuracy/Validation", val_acc,  epoch)

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save({
            "epoch":                epoch,
            "model_state_dict":     PRETRAINED_MODEL.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": lr_scheduler.state_dict(),
            "best_val_loss":        best_val_loss,
            "patience_counter":     patience_counter,
        }, f"models/{model_save_name}.pth")
        print(f"  -> New best model saved (Val Loss: {best_val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

writer.close()